In [28]:
# import h5py
# from tensorflow import keras
# model = keras.models.load_model('Big_Model.h5')
# model.summary()

In [1]:
# Import AI stuff and load data
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")

(train_df.shape, test_df.shape)

((2109600, 8), (451440, 8))

In [20]:
# clean up values

def fill_with_local_mean(series):
    values = series.values.copy()

    for i in range(len(values)):
        if pd.isna(values[i]):
            neighbors = []

            # Previous 2
            if i-2 >= 0:
                neighbors.append(values[i-2])
            if i-1 >= 0:
                neighbors.append(values[i-1])

            # Next 2
            if i+1 < len(values):
                neighbors.append(values[i+1])
            if i+2 < len(values):
                neighbors.append(values[i+2])

            # Remove NaNs from neighbors
            neighbors = [x for x in neighbors if not pd.isna(x)]

            if len(neighbors) > 0:
                values[i] = np.mean(neighbors)

    return pd.Series(values, index=series.index)

train_df["oxygen_saturation"] = fill_with_local_mean(train_df["oxygen_saturation"])
train_df["respiratory_rate"] = fill_with_local_mean(train_df["respiratory_rate"])
train_df["heart_rate"] = fill_with_local_mean(train_df["heart_rate"])
train_df["systolic_bp"] = fill_with_local_mean(train_df["systolic_bp"])
train_df["diastolic_bp"] = fill_with_local_mean(train_df["diastolic_bp"])

train_df[train_df.isna().any(axis=1)]

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
3122,2025-01-01 23:20:10,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3123,2025-01-01 23:20:15,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3124,2025-01-01 23:20:20,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3125,2025-01-01 23:20:25,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
3126,2025-01-01 23:20:30,7a0ccfea-7f8a-fc6e-35fa-3775b13950da_4,0.0,0.0,0.0,0.00,0.0,0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2108154,2025-05-03 18:59:30,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,1.25,0.0,3,0.0,0.0,NaN
2108155,2025-05-03 18:59:35,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,0.00,0.0,3,0.0,0.0,NaN
2108156,2025-05-03 18:59:40,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,5.00,0.0,3,0.0,0.0,NaN
2108157,2025-05-03 18:59:45,3ee6bbfa-9b63-6790-505e-6200cf762b7d_2927,0.0,0.0,0.0,5.00,0.0,3,0.0,0.0,NaN


In [30]:
# Get rid of any useless cols that doesnt help prediction
train_df['pulse_pressure'] = train_df['systolic_bp'] - train_df['diastolic_bp']
train_df['MAP'] = (2 * (train_df['diastolic_bp'] + train_df['systolic_bp'])) / 3 
train_df['shock_index'] = train_df['heart_rate'] / train_df['systolic_bp']
train_df['shock_index'] = train_df['shock_index'].fillna(0)


train_df.head()


,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
0,2025-01-01 19:00:00,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,88.0,104.0,74.0,18.0,95.0,0,30.0,118.666667,0.846154
1,2025-01-01 19:00:05,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,88.0,100.0,72.0,18.0,95.0,0,28.0,114.666667,0.880000
2,2025-01-01 19:00:10,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,86.0,96.0,70.0,17.0,94.0,0,26.0,110.666667,0.895833
3,2025-01-01 19:00:15,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,89.0,98.0,70.0,18.0,94.0,0,28.0,112.000000,0.908163
4,2025-01-01 19:00:20,b317e7ee-8af7-3e9c-3e0f-646395b8c81a_0,91.0,98.0,70.0,18.5,94.0,0,28.0,112.000000,0.928571


In [31]:
# Get rid of any entries with null vals
train_df = train_df.sort_values(["encounter_id", "timestamp"])
train_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index
1577520,2025-04-03 02:00:00,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,75.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.681818
1577521,2025-04-03 02:00:05,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,78.0,110.0,58.0,13.0,98.0,0,52.0,112.000000,0.709091
1577522,2025-04-03 02:00:10,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,83.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.754545
1577523,2025-04-03 02:00:15,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,77.0,110.0,58.5,16.0,97.0,0,51.5,112.333333,0.700000
1577524,2025-04-03 02:00:20,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,79.0,108.0,58.0,17.0,97.0,0,50.0,110.666667,0.731481


In [35]:
# add time based data
window_size = 12  # 1 minute

train_df["hr_roll_mean_60s"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .rolling(window_size, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

train_df["spo2_roll_mean_60s"] = (
    train_df.groupby("encounter_id")["oxygen_saturation"]
      .rolling(window_size, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

train_df["hr_roll_std_60s"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .rolling(window_size, min_periods=1)
      .std()
      .reset_index(level=0, drop=True)
)

train_df["hr_delta"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .diff()
)

train_df["spo2_delta"] = (
    train_df.groupby("encounter_id")["oxygen_saturation"]
      .diff()
)


def rolling_slope(series, window=12):
    slopes = [np.nan] * len(series)

    for i in range(window - 1, len(series)):
        y = series[i-window+1:i+1]
        x = np.arange(len(y))
        slope = np.polyfit(x, y, 1)[0]
        slopes[i] = slope

    return slopes

train_df["hr_slope_60s"] = (
    train_df.groupby("encounter_id")["heart_rate"]
      .transform(lambda x: rolling_slope(x.values, window=12))
)

train_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
1577520,2025-04-03 02:00:00,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,75.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.681818,75.000000,97.000000,NaN,NaN,NaN,NaN
1577521,2025-04-03 02:00:05,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,78.0,110.0,58.0,13.0,98.0,0,52.0,112.000000,0.709091,76.500000,97.500000,2.121320,3.0,1.0,NaN
1577522,2025-04-03 02:00:10,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,83.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.754545,78.666667,97.333333,4.041452,5.0,-1.0,NaN
1577523,2025-04-03 02:00:15,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,77.0,110.0,58.5,16.0,97.0,0,51.5,112.333333,0.700000,78.250000,97.250000,3.403430,-6.0,0.0,NaN
1577524,2025-04-03 02:00:20,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,79.0,108.0,58.0,17.0,97.0,0,50.0,110.666667,0.731481,78.400000,97.200000,2.966479,2.0,0.0,NaN


In [36]:
train_df[["hr_delta", "spo2_delta", "hr_roll_std_60s", "hr_slope_60s"]] = \
train_df[["hr_delta", "spo2_delta", "hr_roll_std_60s", "hr_slope_60s"]].fillna(0)

train_df.head()

,timestamp,encounter_id,heart_rate,systolic_bp,diastolic_bp,respiratory_rate,oxygen_saturation,label,pulse_pressure,MAP,shock_index,hr_roll_mean_60s,spo2_roll_mean_60s,hr_roll_std_60s,hr_delta,spo2_delta,hr_slope_60s
1577520,2025-04-03 02:00:00,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,75.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.681818,75.000000,97.000000,0.000000,0.0,0.0,0.0
1577521,2025-04-03 02:00:05,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,78.0,110.0,58.0,13.0,98.0,0,52.0,112.000000,0.709091,76.500000,97.500000,2.121320,3.0,1.0,0.0
1577522,2025-04-03 02:00:10,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,83.0,110.0,58.0,15.0,97.0,0,52.0,112.000000,0.754545,78.666667,97.333333,4.041452,5.0,-1.0,0.0
1577523,2025-04-03 02:00:15,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,77.0,110.0,58.5,16.0,97.0,0,51.5,112.333333,0.700000,78.250000,97.250000,3.403430,-6.0,0.0,0.0
1577524,2025-04-03 02:00:20,0027a2c3-6f86-b683-cac4-5ac409c14b13_2191,79.0,108.0,58.0,17.0,97.0,0,50.0,110.666667,0.731481,78.400000,97.200000,2.966479,2.0,0.0,0.0


In [7]:
# Create training and testing sets - also separate features/labels
X_train = train_df.drop(columns = ["label"])
y_train = train_df["label"]

X_test = test_df.drop(columns = ["label"])
y_test = test_df["label"]

In [8]:
from tensorflow.keras.utils import to_categorical

unique_states = sorted(y_train.unique())
num_classes = len(unique_states)

# Create a mapping from np.int to actual int
state_to_index = {state: i for i, state in enumerate(unique_states)}

# Convert our training labels to numerical indices
y_train_indices = y_train.map(state_to_index)

# One-hot encode the training labels
y_train_encoded = to_categorical(y_train_indices, num_classes=num_classes)
print("y_train_encoded shape:", y_train_encoded.shape, 
      "Example one-hot vector:", y_train_encoded[0])


# Repeat for testing data
unique_states = sorted(y_test.unique())
num_classes = len(unique_states)

state_to_index = {state: i for i, state in enumerate(unique_states)}

y_test_indices = y_test.map(state_to_index)

y_test_encoded = to_categorical(y_test_indices, num_classes=num_classes)
print("y_train_encoded shape:", y_test_encoded.shape, 
      "Example one-hot vector:", y_test_encoded[0])

y_train_encoded shape: (1906801, 4) Example one-hot vector: [1. 0. 0. 0.]
y_train_encoded shape: (408576, 4) Example one-hot vector: [1. 0. 0. 0.]


In [9]:
model = Sequential()

# Input layer: our features are the vitals
model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))

# Hidden layer 2
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

# Hidden layer 3
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.2))

# Output layer for multi-class
model.add(Dense(num_classes, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,964 (35.02 KB)

 Trainable params: 8,964 (35.02 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
EPOCHS = 10
BATCH_SIZE = 512

history = model.fit(
    X_train,
    y_train_encoded,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,  # 90% train, 10% validation
    verbose=1
)

Epoch 1/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.7562 - loss: 0.6644 - val_accuracy: 0.7971 - val_loss: 0.5734
Epoch 2/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7803 - loss: 0.5521 - val_accuracy: 0.7987 - val_loss: 0.5720
Epoch 3/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7839 - loss: 0.5401 - val_accuracy: 0.7915 - val_loss: 0.6113
Epoch 4/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.7876 - loss: 0.5268 - val_accuracy: 0.7626 - val_loss: 0.7071
Epoch 5/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.7923 - loss: 0.5136 - val_accuracy: 0.6067 - val_loss: 0.9288
Epoch 6/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.7947 - loss: 0.5064 - val_accuracy: 0.5158 - val_loss: 1.0662
Epoch 7/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.7961 - loss: 0.5029 - val_accuracy: 0.3906 - val_loss: 1.1930
Epoch 8/10
3352/3352 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.7973 - loss: 0.5001 - 

In [11]:
model.evaluate(X_test, y_test_encoded)

12768/12768 ━━━━━━━━━━━━━━━━━━━━ 9s 711us/step - accuracy: 0.2837 - loss: 1.3129


[1.31289803981781, 0.283663272857666]